In [1]:
!pip install presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

print("Presidio is ready!")

Presidio is ready!


In [3]:
agent_output = """
Your order has been registered successfully.

Customer name: Ahmed Ali
Email: ahmed.ali@gmail.com
Phone: 0501234567

Your order will arrive tomorrow.
"""

print("========== RAW AGENT OUTPUT ==========")
print(agent_output)

========== RAW AGENT OUTPUT ==========

Your order has been registered successfully.

Customer name: Ahmed Ali
Email: ahmed.ali@gmail.com
Phone: 0501234567

Your order will arrive tomorrow.



In [4]:
results = analyzer.analyze(
    text=agent_output,
    language="en",
    entities=[
        "PERSON",
        "EMAIL_ADDRESS",
        "PHONE_NUMBER"
    ]
)

results = [
    r for r in results
    if r.entity_type in ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"]
    and r.start < r.end
]

print("========== DETECTED PII ==========")

for result in results:
    entity = agent_output[result.start:result.end]

    print(
        f"{result.entity_type}: {entity} "
        f"(score={result.score:.2f})"
    )

========== DETECTED PII ==========
EMAIL_ADDRESS: ahmed.ali@gmail.com (score=1.00)
PERSON: Ahmed Ali
Email (score=0.85)
PHONE_NUMBER: 0501234567 (score=0.75)


In [5]:
protected_output = anonymizer.anonymize(
    text=agent_output,
    analyzer_results=results
)

print("========== PROTECTED OUTPUT ==========")
print(protected_output.text)

========== PROTECTED OUTPUT ==========

Your order has been registered successfully.

Customer name: <PERSON>: <EMAIL_ADDRESS>
Phone: <PHONE_NUMBER>

Your order will arrive tomorrow.



In [6]:
def pii_output_guardrail(agent_output):

    results = analyzer.analyze(
        text=agent_output,
        language="en",
        entities=[
            "PERSON",
            "EMAIL_ADDRESS",
            "PHONE_NUMBER"
        ]
    )

    results = [
        r for r in results
        if r.entity_type in ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"]
        and r.start < r.end
    ]

    if not results:
        return agent_output

    protected_output = anonymizer.anonymize(
        text=agent_output,
        analyzer_results=results
    )

    return protected_output.text

In [7]:
safe_output = pii_output_guardrail(agent_output)

print("========== SAFE AGENT OUTPUT ==========")
print(safe_output)

========== SAFE AGENT OUTPUT ==========

Your order has been registered successfully.

Customer name: <PERSON>: <EMAIL_ADDRESS>
Phone: <PHONE_NUMBER>

Your order will arrive tomorrow.

